In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import random


In [2]:
def generate_polynomial_data(n_points=100, noise_std=0.2, seed=42):
    """
    Generates synthetic (x, y) data for a polynomial y = 3x^3 + 2x^2 - 4x + 1 with added Gaussian noise.
    
    Args:
        n_points (int): Number of data points to generate.
        noise_std (float): Standard deviation of the Gaussian noise.
        seed (int): Random seed for reproducibility.

    Returns:
        (x, y): Tuple of Tensors representing inputs and polynomial outputs.
    """
    # Set random seed for reproducibility
    torch.manual_seed(seed)
    
    # Generate equally spaced x values
    x = torch.linspace(-3, 3, n_points).unsqueeze(1)  # shape: (n_points, 1)

    # Define polynomial: 3x^3 + 2x^2 - 4x + 1
    y_true = 3 * x**3 + 2 * x**2 - 4 * x + 1
    
    # Add Gaussian noise
    noise = torch.randn_like(y_true) * noise_std
    
    # Final y = polynomial + noise
    y = y_true + noise
    
    return x, y


In [3]:
x, y = generate_polynomial_data(n_points=1000, noise_std=0.5)
x = x.to(dtype=torch.float64)
y = y.to(dtype=torch.float64)

In [4]:
def generate_polynomial_data_new(n_points=100, noise_std=0.2, seed=42):
    """
    Generates synthetic (x, y) data for a polynomial y = 3x^3 + 2x^2 - 4x + 2 
    with added Gaussian noise.
    
    Args:
        n_points (int): Number of data points to generate.
        noise_std (float): Standard deviation of the Gaussian noise.
        seed (int): Random seed for reproducibility.

    Returns:
        (x, y): Tuple of Tensors representing inputs and polynomial outputs.
    """
    # Set random seed for reproducibility
    torch.manual_seed(seed)
    
    # Generate equally spaced x values
    x = torch.linspace(-3, 3, n_points).unsqueeze(1)  # shape: (n_points, 1)

    # Define polynomial: 3x^3 + 2x^2 - 4x + 2
    y_true = 3 * x**3 + 2 * x**2 - 4 * x + 2
    
    # Add Gaussian noise
    noise = torch.randn_like(y_true) * noise_std
    
    # Final y = polynomial + noise
    y = y_true + noise
    
    return x, y

In [5]:
x_new, y_new = generate_polynomial_data_new(n_points=1000, noise_std=0.75)
x_new = x_new.to(dtype=torch.float64)
y_new = y_new.to(dtype=torch.float64)

In [6]:
class ThreeLayerNet(nn.Module):
    def __init__(self):
        super(ThreeLayerNet, self).__init__()
        # Define a simple feed-forward network:
        # Input -> Hidden1 -> Hidden2 -> Output
        self.net = nn.Sequential(
            nn.Linear(1, 16),   # Input is 1-D -> 16 neurons
            nn.ReLU(),
            nn.Linear(16, 16),  # 16 neurons -> 16 neurons
            nn.ReLU(),
            nn.Linear(16, 1)    # 16 neurons -> 1-D output
        )
    
    def forward(self, x):
        return self.net(x)

In [7]:
model = ThreeLayerNet()
model = model.to(dtype=torch.float64)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.02)

In [8]:
# Training loop
for epoch in range(100):
    optimizer.zero_grad()
    y_pred = model(x)

    loss = criterion(y_pred, y)

    loss.backward()
    optimizer.step()
    #print(f'loss: {loss.item()}')

Online learning

In [9]:
y_pred = model(x)
loss = criterion(y_pred, y)
print(f'loss: {loss.item()}')

loss: 19.194271217289923


In [10]:
y_pred = model(x_new)
loss = criterion(y_pred, y_new)
print(f'loss: {loss.item()}')

loss: 20.416736024859283


In [11]:
class OModelAutograd(nn.Module):
    def __init__(self, model: nn.Module, params: nn.ParameterList, criterion=nn.MSELoss(), lr=0.001):
        """
        Initialize the optimizer-like module.

        model: The model to optimize (used for forward computation).
        params: A ParameterList representing the parameters to be updated.
        criterion: Loss function (default is MSELoss).
        lr: Learning rate for manual updates.
        """
        super(OModelAutograd, self).__init__()
        self.model = model
        self.params = params  # ParameterList for TorchScript compatibility
        self.criterion = criterion
        self.lr = lr

    def forward_fn(self, x: torch.Tensor) -> torch.Tensor:
        """
        Perform the forward pass using the provided model.
        """
        return self.model(x)

    def forward(self, x: torch.Tensor, x_prev: torch.Tensor, y_prev: torch.Tensor) -> torch.Tensor:
        """
        Perform a forward pass, compute the loss, manually update the parameters,
        and return the updated prediction.

        x: Input tensor
        x_prev: Previous input tensor (for loss calculation)
        y_prev: Target tensor (for loss calculation)
        """
        # 1. Forward pass
        pred = self.forward_fn(x_prev)
        loss = self.criterion(pred, y_prev)

        # 2. Compute gradients for the parameters
        grads = torch.autograd.grad(
            loss,
            self.params,
            create_graph=False,
            retain_graph=False,
            allow_unused=False
        )

        # 3. Manual gradient descent step using a static approach
        with torch.no_grad():
            for param, grad in zip(self.params, grads):
                if grad is not None:  # Ensure gradient is valid
                    param -= self.lr * grad

        # 4. Return updated prediction
        return self.forward_fn(x)


In [12]:
# Create an instance of the OModelAutograd class
new_model = OModelAutograd(
    model=model,
    params=nn.ParameterList(model.parameters()),  # Wrap model parameters in ParameterList
    criterion=nn.MSELoss(),  # Specify the loss function
    lr=0.001  # Learning rate
)

# Example training loop
for _ in range(10):
    xp = torch.zeros(size=(1, 1)).to(dtype=torch.float64)
    yp = torch.zeros(size=(1, 1)).to(dtype=torch.float64)

    # Process data individually
    for xi, yi in zip(x_new, y_new):
        xi = xi.unsqueeze(0)  # Add batch dimension
        yi = yi.unsqueeze(0)  # Add batch dimension

        y_pred = new_model(xi, xp, yp)  # Forward pass
        loss = new_model.criterion(y_pred, yi)  # Compute loss

        # Do swap
        xp = xi
        yp = yi

    # Evaluate on the whole batch
    xp = torch.zeros_like(x_new)
    yp = torch.zeros_like(y_new)
    y_pred = new_model(x_new, xp, yp)
    loss = criterion(y_pred, y_new)  # Batch loss
    print(f'Loss: {loss.item()}')

    # Shuffle x_new and y_new
    data = list(zip(x_new, y_new))  # Combine into pairs
    random.shuffle(data)  # Shuffle pairs
    x_new, y_new = zip(*data)  # Unzip back

    # Convert back to tensors if necessary
    x_new = torch.stack(x_new)
    y_new = torch.stack(y_new)



Loss: 1791.4268297922288
Loss: 636.4583341029721
Loss: 628.5717099296508
Loss: 628.4716011141214
Loss: 628.579077742786
Loss: 628.5627993046905
Loss: 628.4782377604515
Loss: 628.6801816436266
Loss: 628.4732995060398
Loss: 628.5081624246812


Save as script

In [15]:
# Example inputs for tracing with batch size of 1
x = torch.randn(1, 1).to(dtype=torch.float64)  # Current input tensor with batch size 1
x_prev = torch.randn(1, 1).to(dtype=torch.float64)   # Previous input tensor with batch size 1
y_prev = torch.randn(1, 1).to(dtype=torch.float64)    # Target tensor with batch size 1

# Trace the model
script_model = torch.jit.trace(new_model, (x, x_prev, y_prev))
script_model.save(f"./online_example.pt")


RuntimeError: Cannot insert a Tensor that requires grad as a constant. Consider making it a parameter or input, or detaching the gradient
Tensor:
-0.5140
 1.2839
-0.4535
 1.3365
-0.8070
 0.7299
-0.0751
 3.5298
-0.0927
 0.0926
-0.2778
 1.9081
 1.9370
 1.6796
 4.0997
 2.7750
[ torch.DoubleTensor{16,1} ]